In [6]:
# -*- coding: utf-8 -*-
import os
import json
import numpy as np
import pandas as pd
import xgboost as xgb
import shap
import matplotlib.pyplot as plt

# =========================
# 0) 配置
# =========================
RANDOM_SEED = 42
FEATURE_FILE = "./Malodors_Rule&FG&Morgan&StructKG_features.xlsx"

OUT_DIR = "./shap_cache_and_plots_24labels"
CACHE_DIR = os.path.join(OUT_DIR, "cache")
PLOT_DIR  = os.path.join(OUT_DIR, "plots")
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(PLOT_DIR, exist_ok=True)

DPI = 600
TOPK_FINAL = 24   # 热图 / 散点图最终显示 Top24 原始特征

# 24个要训练的标签
TARGET_LABELS_24 = [
    "alcoholic", "aldehydic", "almond", "aromatic", "burnt", "cabbage",
    "cheesy", "cherry", "chocolate", "ethereal", "fishy", "fruity",
    "garlic", "grassy", "green", "ketonic", "musty", "pungent",
    "sharp", "solvent", "sour", "sulfurous", "sweaty", "sweet"
]

# 标签列名：只用于从 X 中排除，避免标签列被当成特征列
LABELS_138 = [
    "alcoholic", "aldehydic", "almond", "aromatic", "burnt", "cabbage",
    "cheesy", "cherry", "chocolate", "ethereal", "fishy", "fruity",
    "garlic", "grassy", "green", "ketonic", "musty", "pungent",
    "sharp", "solvent", "sour", "sulfurous", "sweaty", "sweet"
]

# 最优超参
BEST_PARAMS = {
    "n_estimators": 433,
    "max_depth": 7,
    "learning_rate": 0.0350057872293877,
    "subsample": 0.9947153135691092,
    "colsample_bytree": 0.7778835626400454,
    "min_child_weight": 1.0072775841844182,
    "reg_lambda": 3.4681854273849724,
    "reg_alpha": 6.955456414716767e-08,
    "gamma": 3.606069985094933
}

BASE_XGB_PARAMS_SINGLE = dict(
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",   # 有GPU可改为 gpu_hist
    n_jobs=-1,
    random_state=RANDOM_SEED,
    verbosity=0,
)

# 缓存 SHAP 用 float16 压缩，减少体积
SHAP_SAVE_DTYPE = np.float16


# =========================
# 1) X / y 构建
# =========================
def find_smiles_col(df: pd.DataFrame):
    cand = [c for c in df.columns if isinstance(c, str) and "smiles" in c.lower()]
    if not cand:
        return None

    for p in ["Canonical SMILES", "canonical_smiles", "SMILES", "smiles", "StdSMILES"]:
        for c in cand:
            if c.lower() == p.lower():
                return c

    return cand[0]


def is_numeric_or_convertible(series: pd.Series) -> bool:
    if np.issubdtype(series.dtype, np.number) or series.dtype == bool:
        return True

    try:
        pd.to_numeric(series, errors="raise")
        return True
    except Exception:
        return False


def build_X_y(df: pd.DataFrame):
    smiles_col = find_smiles_col(df)

    # y24
    miss24 = [c for c in TARGET_LABELS_24 if c not in df.columns]
    if miss24:
        raise ValueError(f"缺少24个目标标签列：{miss24}")

    y24 = df[TARGET_LABELS_24].fillna(0).astype(int).values

    # X：仅排除标签列和 SMILES 列，不删除任何真实特征
    exclude = set([c for c in LABELS_138 if c in df.columns])

    if smiles_col is not None:
        exclude.add(smiles_col)

    feat_cols = [c for c in df.columns if c not in exclude]

    # 去掉无法转为数值的非特征列
    bad = []
    feat_cols2 = []

    for c in feat_cols:
        if is_numeric_or_convertible(df[c]):
            feat_cols2.append(c)
        else:
            bad.append(c)

    if bad:
        print(
            f"[WARN] Dropped non-numeric X columns ({len(bad)}):",
            bad[:10],
            "..." if len(bad) > 10 else ""
        )

    X_df = df[feat_cols2].copy()

    for c in X_df.columns:
        if not (np.issubdtype(X_df[c].dtype, np.number) or X_df[c].dtype == bool):
            X_df[c] = pd.to_numeric(X_df[c], errors="coerce")

    X = X_df.fillna(0).astype(np.float32).values

    return X, y24, feat_cols2, smiles_col


# =========================
# 2) 训练 + SHAP + 缓存
# =========================
def train_one_label(X, y_bin):
    params = dict(BASE_XGB_PARAMS_SINGLE)
    params.update(BEST_PARAMS)

    clf = xgb.XGBClassifier(**params)
    clf.fit(X, y_bin)

    return clf


def signed_meanabs_from_shap(sv: np.ndarray):
    """
    计算带符号的 mean(|SHAP|)。
    绝对值表示贡献强度，符号由 SHAP 总和方向决定。
    """
    meanabs = np.mean(np.abs(sv), axis=0)
    sgn = np.sign(np.sum(sv, axis=0))
    sgn[sgn == 0] = 1.0

    return meanabs * sgn


def save_beeswarm(shap_vals, X_vals, feature_names, out_png, max_display=24):
    plt.figure(figsize=(9.0, 11.0))

    shap.summary_plot(
        shap_vals,
        X_vals,
        feature_names=feature_names,
        plot_type="dot",
        max_display=max_display,
        show=False
    )

    plt.gcf().subplots_adjust(left=0.38, right=0.98, top=0.98, bottom=0.08)
    plt.savefig(out_png, dpi=DPI, bbox_inches="tight")
    plt.close()


def plot_heatmap_signed(matrix, row_names, col_names, out_png):
    plt.figure(
        figsize=(
            max(10, 0.55 * len(col_names)),
            max(6, 0.40 * len(row_names))
        )
    )

    im = plt.imshow(matrix, aspect="auto", cmap="bwr")

    plt.colorbar(
        im,
        label="Signed mean(|SHAP|)  (sign = sign(sum SHAP))"
    )

    plt.xticks(
        np.arange(len(col_names)),
        col_names,
        rotation=90,
        fontsize=9
    )

    plt.yticks(
        np.arange(len(row_names)),
        row_names,
        fontsize=10
    )

    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            plt.text(
                j,
                i,
                f"{matrix[i, j]:+.2f}",
                ha="center",
                va="center",
                fontsize=6
            )

    plt.tight_layout()
    plt.savefig(out_png, dpi=DPI, bbox_inches="tight")
    plt.close()


# =========================
# 3) 主流程
# =========================
def main():
    print("[INFO] Reading:", FEATURE_FILE)

    df = pd.read_excel(FEATURE_FILE)

    X, y24, feat_cols, smiles_col = build_X_y(df)

    print(f"[INFO] X={X.shape}, y24={y24.shape}, features={len(feat_cols)}")

    if smiles_col:
        print("[INFO] smiles_col =", smiles_col)

    # 保存 X 与特征名
    np.savez_compressed(
        os.path.join(CACHE_DIR, "X_full.npz"),
        X=X.astype(np.float16)
    )

    with open(os.path.join(CACHE_DIR, "feature_names.json"), "w", encoding="utf-8") as f:
        json.dump(feat_cols, f, ensure_ascii=False, indent=2)

    meta = {
        "feature_file": FEATURE_FILE,
        "n_samples": int(X.shape[0]),
        "n_features": int(X.shape[1]),
        "target_labels_24": TARGET_LABELS_24,
        "smiles_col": smiles_col,
        "best_params": BEST_PARAMS,
        "base_xgb_params_single": BASE_XGB_PARAMS_SINGLE,
        "topk_final": TOPK_FINAL,
        "feature_display_mode": "original_features_only_no_drop_no_aggregation"
    }

    with open(os.path.join(CACHE_DIR, "meta.json"), "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)

    # ============ A) 训练24个模型 + 计算并缓存全量 SHAP ============
    meanabs_24xF = np.zeros(
        (len(TARGET_LABELS_24), X.shape[1]),
        dtype=np.float32
    )

    signed_24xF = np.zeros(
        (len(TARGET_LABELS_24), X.shape[1]),
        dtype=np.float32
    )

    shap_full_list = []

    for li, lab in enumerate(TARGET_LABELS_24):
        print(f"\n[TRAIN+SHAP] {lab} ({li + 1}/{len(TARGET_LABELS_24)})")

        y_bin = y24[:, li]

        model = train_one_label(X, y_bin)

        model_path = os.path.join(CACHE_DIR, f"xgb__{lab}.json")
        model.get_booster().save_model(model_path)

        explainer = shap.TreeExplainer(model)
        sv = explainer.shap_values(X)  # shape: (n_samples, n_features)

        sv = np.asarray(sv, dtype=np.float32)

        shap_full_list.append(sv)

        meanabs = np.mean(np.abs(sv), axis=0).astype(np.float32)
        signed = signed_meanabs_from_shap(sv).astype(np.float32)

        meanabs_24xF[li, :] = meanabs
        signed_24xF[li, :] = signed

        # 缓存全量 SHAP
        np.savez_compressed(
            os.path.join(CACHE_DIR, f"shap_full__{lab}.npz"),
            shap_values=sv.astype(SHAP_SAVE_DTYPE)
        )

        np.save(
            os.path.join(CACHE_DIR, f"meanabs_full__{lab}.npy"),
            meanabs
        )

    np.savez_compressed(
        os.path.join(CACHE_DIR, "meanabs_24xF.npz"),
        meanabs_24xF=meanabs_24xF,
        labels=np.array(TARGET_LABELS_24)
    )

    np.savez_compressed(
        os.path.join(CACHE_DIR, "signed_24xF.npz"),
        signed_24xF=signed_24xF,
        labels=np.array(TARGET_LABELS_24)
    )

    print("\n[SAVED] All models + full SHAP cache done.")

    # ============ B) 直接基于原始特征选择 Top24 ============
    global_meanabs = meanabs_24xF.mean(axis=0)

    top_idx = np.argsort(global_meanabs)[::-1][:TOPK_FINAL]
    top_names = [feat_cols[i] for i in top_idx]

    pd.DataFrame({
        "rank": np.arange(1, TOPK_FINAL + 1),
        "feature": top_names,
        "global_mean_abs_shap": global_meanabs[top_idx]
    }).to_csv(
        os.path.join(CACHE_DIR, f"Top{TOPK_FINAL}_original_features.csv"),
        index=False,
        encoding="utf-8-sig"
    )

    print(f"[SAVED] Top{TOPK_FINAL}_original_features.csv")

    # ============ C) 热图：24 labels × Top24 原始特征 ============
    heat = signed_24xF[:, top_idx]

    heat_png = os.path.join(
        PLOT_DIR,
        f"Heatmap_24labels_Top{TOPK_FINAL}_original_features.png"
    )

    plot_heatmap_signed(
        heat,
        TARGET_LABELS_24,
        top_names,
        heat_png
    )

    pd.DataFrame(
        heat,
        index=TARGET_LABELS_24,
        columns=top_names
    ).to_csv(
        os.path.join(
            PLOT_DIR,
            f"Heatmap_24labels_Top{TOPK_FINAL}_original_features.csv"
        ),
        encoding="utf-8-sig"
    )

    print("[SAVED]", heat_png)

    # ============ D) Beeswarm：每个标签一张，直接显示 Top24 原始特征 ============
    X_top = X[:, top_idx]

    for li, lab in enumerate(TARGET_LABELS_24):
        sv_full = shap_full_list[li]
        sv_top = sv_full[:, top_idx]

        out_png = os.path.join(
            PLOT_DIR,
            f"Beeswarm__{lab}__Top{TOPK_FINAL}_original_features.png"
        )

        save_beeswarm(
            sv_top,
            X_top,
            top_names,
            out_png,
            max_display=TOPK_FINAL
        )

        # 缓存每个标签的 Top24 SHAP，后续画图可直接读取
        np.savez_compressed(
            os.path.join(
                CACHE_DIR,
                f"shap_original_top{TOPK_FINAL}__{lab}.npz"
            ),
            shap_values=sv_top.astype(SHAP_SAVE_DTYPE)
        )

    print("\n[DONE] Everything finished.")
    print("Outputs:")
    print(" - Cache:", CACHE_DIR)
    print(" - Plots:", PLOT_DIR)


if __name__ == "__main__":
    main()

[INFO] Reading: ./Malodors_Rule&FG&Morgan&StructKG_features.xlsx
[INFO] X=(3756, 2595), y24=(3756, 24), features=2595
[INFO] smiles_col = Canonical_SMILES

[TRAIN+SHAP] alcoholic (1/24)

[TRAIN+SHAP] aldehydic (2/24)

[TRAIN+SHAP] almond (3/24)

[TRAIN+SHAP] aromatic (4/24)

[TRAIN+SHAP] burnt (5/24)

[TRAIN+SHAP] cabbage (6/24)

[TRAIN+SHAP] cheesy (7/24)

[TRAIN+SHAP] cherry (8/24)

[TRAIN+SHAP] chocolate (9/24)

[TRAIN+SHAP] ethereal (10/24)

[TRAIN+SHAP] fishy (11/24)

[TRAIN+SHAP] fruity (12/24)

[TRAIN+SHAP] garlic (13/24)

[TRAIN+SHAP] grassy (14/24)

[TRAIN+SHAP] green (15/24)

[TRAIN+SHAP] ketonic (16/24)

[TRAIN+SHAP] musty (17/24)

[TRAIN+SHAP] pungent (18/24)

[TRAIN+SHAP] sharp (19/24)

[TRAIN+SHAP] solvent (20/24)

[TRAIN+SHAP] sour (21/24)

[TRAIN+SHAP] sulfurous (22/24)

[TRAIN+SHAP] sweaty (23/24)

[TRAIN+SHAP] sweet (24/24)

[SAVED] All models + full SHAP cache done.
[SAVED] Top24_original_features.csv
[SAVED] ./shap_cache_and_plots_24labels/plots/Heatmap_24labels_Top

In [7]:
# -*- coding: utf-8 -*-
import os, re, json
import numpy as np
import pandas as pd
import xgboost as xgb
import shap
import matplotlib.pyplot as plt

# =========================
# 0) 配置
# =========================
RANDOM_SEED = 42
FEATURE_FILE = "./Malodors_Rule&FG&Morgan&StructKG_features.xlsx"

OUT_DIR = "./shap_cache_and_plots_24labels_fulltrain_plotFilter"
CACHE_DIR = os.path.join(OUT_DIR, "cache")
PLOT_DIR  = os.path.join(OUT_DIR, "plots")
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(PLOT_DIR, exist_ok=True)

DPI = 1000

# 热图显示特征数
TOPK_HEATMAP = 24

# 单个气味 beeswarm 显示特征数
TOPK_BEESWARM = 12

# 是否仅在单个 beeswarm 图中隐藏 Morgan / StructKG
# 当前不隐藏任何特征，beeswarm 也从全部特征中选择 TopK
FILTER_MORGAN_STRUCTKG_IN_BEESWARM = False

TARGET_LABELS_24 = [
    "alcoholic", "aldehydic", "almond", "aromatic", "burnt", "cabbage",
    "cheesy", "cherry", "chocolate", "ethereal", "fishy", "fruity",
    "garlic", "grassy", "green", "ketonic", "musty", "pungent",
    "sharp", "solvent", "sour", "sulfurous", "sweaty", "sweet"
]

LABELS_138 = [
    "alcoholic", "aldehydic", "almond", "aromatic", "burnt", "cabbage",
    "cheesy", "cherry", "chocolate", "ethereal", "fishy", "fruity",
    "garlic", "grassy", "green", "ketonic", "musty", "pungent",
    "sharp", "solvent", "sour", "sulfurous", "sweaty", "sweet"
]

BEST_PARAMS = {
    "n_estimators": 433,
    "max_depth": 7,
    "learning_rate": 0.0350057872293877,
    "subsample": 0.9947153135691092,
    "colsample_bytree": 0.7778835626400454,
    "min_child_weight": 1.0072775841844182,
    "reg_lambda": 3.4681854273849724,
    "reg_alpha": 6.955456414716767e-08,
    "gamma": 3.606069985094933
}

BASE_XGB_PARAMS_SINGLE = dict(
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    n_jobs=-1,
    random_state=RANDOM_SEED,
    verbosity=0,
)

SHAP_SAVE_DTYPE = np.float16


# =========================
# 1) X / y 构建
# =========================
def find_smiles_col(df: pd.DataFrame):
    cand = [c for c in df.columns if isinstance(c, str) and "smiles" in c.lower()]
    if not cand:
        return None

    for p in ["Canonical SMILES", "canonical_smiles", "SMILES", "smiles", "StdSMILES"]:
        for c in cand:
            if c.lower() == p.lower():
                return c

    return cand[0]


def is_numeric_or_convertible(series: pd.Series) -> bool:
    if np.issubdtype(series.dtype, np.number) or series.dtype == bool:
        return True

    try:
        pd.to_numeric(series, errors="raise")
        return True
    except Exception:
        return False


def build_X_y(df: pd.DataFrame):
    smiles_col = find_smiles_col(df)

    miss24 = [c for c in TARGET_LABELS_24 if c not in df.columns]
    if miss24:
        raise ValueError(f"缺少24个目标标签列：{miss24}")

    y24 = df[TARGET_LABELS_24].fillna(0).astype(int).values

    exclude = set([c for c in LABELS_138 if c in df.columns])

    if smiles_col is not None:
        exclude.add(smiles_col)

    feat_cols = [c for c in df.columns if c not in exclude]

    bad = []
    feat_cols2 = []

    for c in feat_cols:
        if is_numeric_or_convertible(df[c]):
            feat_cols2.append(c)
        else:
            bad.append(c)

    if bad:
        print(
            f"[WARN] Dropped non-numeric X columns ({len(bad)}):",
            bad[:10],
            "..." if len(bad) > 10 else ""
        )

    X_df = df[feat_cols2].copy()

    for c in X_df.columns:
        if not (np.issubdtype(X_df[c].dtype, np.number) or X_df[c].dtype == bool):
            X_df[c] = pd.to_numeric(X_df[c], errors="coerce")

    X = X_df.fillna(0).astype(np.float32).values

    return X, y24, feat_cols2, smiles_col


# =========================
# 2) Morgan / StructKG 列识别
# 仅用于记录信息，不影响训练和绘图筛选
# =========================
def is_morgan_name(col):
    s = str(col).strip().lower()

    if s.isdigit():
        return True

    if re.match(r"^(morgan|ecfp|mfp|fp|bit)[\s_\-]?\d+$", s):
        return True

    if s.startswith(("morgan_", "ecfp_", "fp_", "bit_", "mfp_")):
        return True

    return False


def is_structkg_name(col):
    s = str(col).strip().lower()

    if s.startswith("kg_emb_"):
        return True

    if s.startswith("structkg"):
        return True

    if "structkg" in s:
        return True

    if ("kg" in s) and any(k in s for k in ["emb", "embed", "embedding", "dim"]):
        return True

    return False


def detect_morgan_structkg_cols(feat_cols: list):
    m_cols = [c for c in feat_cols if is_morgan_name(c)]
    k_cols = [c for c in feat_cols if is_structkg_name(c)]
    return m_cols, k_cols


# =========================
# 3) 模型训练与 SHAP
# =========================
def train_one_label(X, y_bin):
    params = dict(BASE_XGB_PARAMS_SINGLE)
    params.update(BEST_PARAMS)

    clf = xgb.XGBClassifier(**params)
    clf.fit(X, y_bin)

    return clf


def signed_meanabs_from_shap(sv: np.ndarray):
    meanabs = np.mean(np.abs(sv), axis=0)
    sgn = np.sign(np.sum(sv, axis=0))
    sgn[sgn == 0] = 1.0
    return meanabs * sgn


import textwrap
import matplotlib.pyplot as plt


# =========================
# 特征名清洗 + 自动换行
# =========================
def wrap_feature_name(s, width=32, max_lines=2):
    """
    将过长特征名自动折行。
    width: 每行大致字符数
    max_lines: 最多显示行数
    """
    s = str(s)

    replacements = {
        "&&": " && ",
        "||": " || ",
        ">=": " ≥ ",
        "<=": " ≤ ",
        "==": " = ",
        ">": " > ",
        "<": " < ",
    }

    for old, new in replacements.items():
        s = s.replace(old, new)

    s = " ".join(s.split())

    lines = textwrap.wrap(
        s,
        width=width,
        break_long_words=True,
        break_on_hyphens=False
    )

    if len(lines) > max_lines:
        lines = lines[:max_lines]
        lines[-1] = lines[-1].rstrip(".") + "..."

    return "\n".join(lines)


def clean_feature_name(name):
    """
    用于绘图时缩短和折叠特征名称。
    不改变原始特征，仅改变图上的显示。
    """
    s = str(name)

    s = s.replace("Rule__", "")
    s = s.replace("FG__", "FG: ")
    s = s.replace("FG: FG:", "FG: ")

    s = s.replace("NumAliphaticCarbons", "Aliphatic C")
    s = s.replace("NumAromaticRings", "Aromatic rings")
    s = s.replace("NumRotatableBonds", "Rotatable bonds")
    s = s.replace("Rotatable_bonds", "Rotatable bonds")
    s = s.replace("H-bond acceptors", "HBA")
    s = s.replace("H-bond donors", "HBD")
    s = s.replace("HasAlcohol", "Alcohol")
    s = s.replace("MolWt", "MW")
    s = s.replace("MolLogP", "LogP")
    s = s.replace("Atom count:", "Atom:")
    s = s.replace("count >=", "≥")

    s = s.replace("(", "(").replace(")", ")")

    s = wrap_feature_name(
        s,
        width=32,
        max_lines=2
    )

    return s


# =========================
# SHAP beeswarm 绘图：字体适中 + 特征名自动换行
# =========================
def save_beeswarm_bold(
    shap_vals,
    X_vals,
    feature_names,
    out_png,
    label_name,
    max_display=12,
    color_bar_label="Feature value"
):
    display_names = [clean_feature_name(x) for x in feature_names]

    plt.figure(figsize=(10.5, 5.8), dpi=DPI)

    try:
        shap.summary_plot(
            shap_vals,
            features=X_vals,
            feature_names=display_names,
            max_display=max_display,
            show=False,
            color_bar_label=color_bar_label,
            cmap=plt.get_cmap("coolwarm"),
            plot_size=None
        )
    except TypeError:
        shap.summary_plot(
            shap_vals,
            features=X_vals,
            feature_names=display_names,
            max_display=max_display,
            show=False,
            cmap=plt.get_cmap("coolwarm"),
            plot_size=None
        )

    fig = plt.gcf()
    ax = fig.axes[0]

    ax.set_title(
        label_name,
        pad=10,
        fontsize=18,
        fontweight="bold"
    )

    ax.set_xlabel(
        "SHAP value",
        fontsize=16,
        fontweight="bold"
    )

    ax.tick_params(axis="x", labelsize=13)

    for lab in ax.get_xticklabels():
        lab.set_fontsize(13)
        lab.set_fontweight("bold")

    ax.tick_params(axis="y", labelsize=11)

    for lab in ax.get_yticklabels():
        lab.set_fontsize(11)
        lab.set_fontweight("bold")
        lab.set_linespacing(0.92)

    ax.axvline(0, color="gray", lw=1.2)

    if len(fig.axes) > 1:
        cax = fig.axes[-1]
        cax.tick_params(labelsize=12)

        try:
            cax.set_ylabel(
                color_bar_label,
                fontsize=14,
                fontweight="bold"
            )
        except Exception:
            pass

        for tl in cax.get_xticklabels() + cax.get_yticklabels():
            tl.set_fontweight("bold")
            tl.set_fontsize(12)

        for spine in cax.spines.values():
            spine.set_linewidth(1.0)

    plt.gcf().subplots_adjust(
        left=0.42,
        right=0.96,
        top=0.88,
        bottom=0.16
    )

    plt.savefig(out_png, dpi=DPI, bbox_inches="tight")
    plt.close()


# =========================
# 5) 热图绘制
# 热图默认使用全部特征排序，可以包含 Morgan / StructKG
# =========================
def plot_heatmap_signed_bold(matrix, row_names, col_names, out_png):
    display_cols = [clean_feature_name(x) for x in col_names]

    fig_w = max(12, 0.70 * len(display_cols))
    fig_h = max(8, 0.45 * len(row_names))

    plt.figure(figsize=(fig_w, fig_h), dpi=DPI)

    vmax = np.nanmax(np.abs(matrix))
    if not np.isfinite(vmax) or vmax == 0:
        vmax = 1.0

    im = plt.imshow(
        matrix,
        aspect="auto",
        cmap="coolwarm",
        vmin=-vmax,
        vmax=vmax
    )

    cbar = plt.colorbar(im)
    cbar.set_label("Signed mean(|SHAP|)", fontsize=18, fontweight="bold")
    cbar.ax.tick_params(labelsize=16)

    for tl in cbar.ax.get_xticklabels() + cbar.ax.get_yticklabels():
        tl.set_fontweight("bold")

    plt.xticks(
        np.arange(len(display_cols)),
        display_cols,
        rotation=90,
        fontsize=14,
        fontweight="bold"
    )

    plt.yticks(
        np.arange(len(row_names)),
        row_names,
        fontsize=15,
        fontweight="bold"
    )

    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            val = matrix[i, j]
            plt.text(
                j,
                i,
                f"{val:+.2f}",
                ha="center",
                va="center",
                fontsize=8,
                fontweight="bold",
                color="black"
            )

    plt.xlabel("Feature", fontsize=18, fontweight="bold")
    plt.ylabel("Odor descriptor", fontsize=18, fontweight="bold")

    plt.tight_layout()
    plt.savefig(out_png, dpi=DPI, bbox_inches="tight")
    plt.close()


# =========================
# 6) 主流程
# =========================
def main():
    print("[INFO] Reading:", FEATURE_FILE)
    df = pd.read_excel(FEATURE_FILE)

    X, y24, feat_cols, smiles_col = build_X_y(df)

    print(f"[INFO] X={X.shape}, y24={y24.shape}, features={len(feat_cols)}")
    if smiles_col:
        print("[INFO] smiles_col =", smiles_col)

    # 识别 Morgan / StructKG
    # 注意：这里只用于输出记录，不用于训练过滤，也不用于 beeswarm 过滤
    m_cols, k_cols = detect_morgan_structkg_cols(feat_cols)

    print(f"[INFO] detected Morgan cols  = {len(m_cols)}")
    print(f"[INFO] detected StructKG cols = {len(k_cols)}")
    print("[INFO] Training features remain FULL. Morgan/StructKG are NOT removed from model training.")
    print("[INFO] Beeswarm features also remain FULL. No feature is hidden in plotting.")

    col_to_i = {c: i for i, c in enumerate(feat_cols)}
    m_idx_all = [col_to_i[c] for c in m_cols if c in col_to_i]
    k_idx_all = [col_to_i[c] for c in k_cols if c in col_to_i]

    plot_filter_set = set(m_idx_all).union(set(k_idx_all))

    # 单个 beeswarm 图候选特征
    # FILTER_MORGAN_STRUCTKG_IN_BEESWARM = False 时，不隐藏任何特征
    if FILTER_MORGAN_STRUCTKG_IN_BEESWARM:
        beeswarm_candidate_idx = [
            i for i in range(len(feat_cols))
            if i not in plot_filter_set
        ]
        hidden_feature_count = len(plot_filter_set)
    else:
        beeswarm_candidate_idx = list(range(len(feat_cols)))
        hidden_feature_count = 0

    print(f"[INFO] beeswarm candidate features = {len(beeswarm_candidate_idx)}")
    print(f"[INFO] features hidden only in beeswarm = {hidden_feature_count}")

    # 保存基础缓存
    np.savez_compressed(
        os.path.join(CACHE_DIR, "X_full.npz"),
        X=X.astype(np.float16)
    )

    with open(os.path.join(CACHE_DIR, "feature_names.json"), "w", encoding="utf-8") as f:
        json.dump(feat_cols, f, ensure_ascii=False, indent=2)

    meta = {
        "feature_file": FEATURE_FILE,
        "n_samples": int(X.shape[0]),
        "n_features_full_training": int(X.shape[1]),
        "target_labels_24": TARGET_LABELS_24,
        "smiles_col": smiles_col,
        "morgan_count_detected": int(len(m_idx_all)),
        "structkg_count_detected": int(len(k_idx_all)),
        "morgan_cols_preview": m_cols[:20],
        "structkg_cols_preview": k_cols[:20],
        "best_params": BEST_PARAMS,
        "base_xgb_params_single": BASE_XGB_PARAMS_SINGLE,
        "topk_heatmap": TOPK_HEATMAP,
        "topk_beeswarm": TOPK_BEESWARM,
        "dpi": DPI,
        "plot_cmap": "coolwarm",
        "filter_morgan_structkg_in_beeswarm": bool(FILTER_MORGAN_STRUCTKG_IN_BEESWARM),
        "hidden_feature_count_in_beeswarm": int(hidden_feature_count),
        "note": (
            "All features, including Morgan and StructKG, are used for model training, "
            "SHAP calculation, heatmap ranking, and individual beeswarm plots. "
            "No feature is removed or hidden in the visualization stage."
        ),
    }

    with open(os.path.join(CACHE_DIR, "meta.json"), "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)

    # ============ A) 训练24个模型 + 计算全量SHAP ============
    meanabs_24xF = np.zeros((len(TARGET_LABELS_24), X.shape[1]), dtype=np.float32)
    signed_24xF = np.zeros((len(TARGET_LABELS_24), X.shape[1]), dtype=np.float32)

    shap_full_list = []

    for li, lab in enumerate(TARGET_LABELS_24):
        print(f"\n[TRAIN+SHAP] {lab} ({li + 1}/{len(TARGET_LABELS_24)})")

        y_bin = y24[:, li]

        model = train_one_label(X, y_bin)

        model_path = os.path.join(CACHE_DIR, f"xgb__{lab}.json")
        model.get_booster().save_model(model_path)

        explainer = shap.TreeExplainer(model)
        sv = explainer.shap_values(X)

        if isinstance(sv, list):
            sv = sv[1]

        sv = np.asarray(sv, dtype=np.float32)

        shap_full_list.append(sv)

        meanabs = np.mean(np.abs(sv), axis=0).astype(np.float32)
        meanabs_24xF[li, :] = meanabs
        signed_24xF[li, :] = signed_meanabs_from_shap(sv).astype(np.float32)

        np.savez_compressed(
            os.path.join(CACHE_DIR, f"shap_full__{lab}.npz"),
            shap_values=sv.astype(SHAP_SAVE_DTYPE),
        )

        np.save(
            os.path.join(CACHE_DIR, f"meanabs_full__{lab}.npy"),
            meanabs
        )

    np.savez_compressed(
        os.path.join(CACHE_DIR, "meanabs_24xF.npz"),
        meanabs_24xF=meanabs_24xF,
        labels=np.array(TARGET_LABELS_24)
    )

    print("\n[SAVED] All models + full SHAP cache done.")

    # ============ B) 热图：默认使用全部特征，包括 Morgan / StructKG ============
    global_meanabs_full = meanabs_24xF.mean(axis=0)

    top_heatmap_idx = np.argsort(global_meanabs_full)[::-1][:TOPK_HEATMAP]
    top_heatmap_names = [feat_cols[i] for i in top_heatmap_idx]

    pd.DataFrame({
        "rank": np.arange(1, len(top_heatmap_idx) + 1),
        "feature": top_heatmap_names,
        "feature_display": [clean_feature_name(x) for x in top_heatmap_names],
        "global_mean_abs_shap": global_meanabs_full[top_heatmap_idx],
        "is_morgan": [int(i in set(m_idx_all)) for i in top_heatmap_idx],
        "is_structkg": [int(i in set(k_idx_all)) for i in top_heatmap_idx],
    }).to_csv(
        os.path.join(CACHE_DIR, f"Top{TOPK_HEATMAP}_features_full_for_heatmap.csv"),
        index=False,
        encoding="utf-8-sig"
    )

    heat = signed_24xF[:, top_heatmap_idx]

    heat_png = os.path.join(
        PLOT_DIR,
        f"Heatmap_24labels_Top{TOPK_HEATMAP}_fullFeatures.png"
    )

    plot_heatmap_signed_bold(
        heat,
        TARGET_LABELS_24,
        top_heatmap_names,
        heat_png
    )

    pd.DataFrame(
        heat,
        index=TARGET_LABELS_24,
        columns=top_heatmap_names
    ).to_csv(
        os.path.join(PLOT_DIR, f"Heatmap_24labels_Top{TOPK_HEATMAP}_fullFeatures.csv"),
        encoding="utf-8-sig"
    )

    print("[SAVED]", heat_png)

    # ============ C) Beeswarm：每个标签从全部特征中选择 Top12 ============
    beeswarm_candidate_idx = np.array(beeswarm_candidate_idx, dtype=int)

    for li, lab in enumerate(TARGET_LABELS_24):
        sv_full = shap_full_list[li]

        meanabs_label = np.mean(np.abs(sv_full), axis=0)

        # 当前不隐藏任何特征，从全部特征中选择该标签的 TopK
        candidate_scores = meanabs_label[beeswarm_candidate_idx]
        order_local = np.argsort(candidate_scores)[::-1][:TOPK_BEESWARM]

        top_beeswarm_idx = beeswarm_candidate_idx[order_local]
        top_beeswarm_names = [feat_cols[i] for i in top_beeswarm_idx]

        sv_top = sv_full[:, top_beeswarm_idx]
        X_top = X[:, top_beeswarm_idx]

        rank_df = pd.DataFrame({
            "rank": np.arange(1, len(top_beeswarm_idx) + 1),
            "feature": top_beeswarm_names,
            "feature_display": [clean_feature_name(x) for x in top_beeswarm_names],
            "mean_abs_shap_this_label": meanabs_label[top_beeswarm_idx],
            "note": "All features are considered in this beeswarm plot."
        })

        rank_csv = os.path.join(
            CACHE_DIR,
            f"Top{TOPK_BEESWARM}_beeswarm_features__{lab}__fullFeatures.csv"
        )
        rank_df.to_csv(rank_csv, index=False, encoding="utf-8-sig")

        out_png = os.path.join(
            PLOT_DIR,
            f"Beeswarm__{lab}__Top{TOPK_BEESWARM}_fullFeatures.png"
        )

        save_beeswarm_bold(
            shap_vals=sv_top,
            X_vals=X_top,
            feature_names=top_beeswarm_names,
            out_png=out_png,
            label_name=lab,
            max_display=TOPK_BEESWARM,
            color_bar_label="Feature value"
        )

        np.savez_compressed(
            os.path.join(CACHE_DIR, f"shap_top{TOPK_BEESWARM}_beeswarm__{lab}__fullFeatures.npz"),
            shap_values=sv_top.astype(SHAP_SAVE_DTYPE),
            feature_idx=top_beeswarm_idx.astype(np.int32),
            feature_names=np.array(top_beeswarm_names, dtype=object),
        )

        print("[SAVED]", out_png)

    print("\n[DONE] Everything finished.")
    print("Outputs:")
    print(" - Cache:", CACHE_DIR)
    print(" - Plots:", PLOT_DIR)


if __name__ == "__main__":
    main()

[INFO] Reading: ./Malodors_Rule&FG&Morgan&StructKG_features.xlsx
[INFO] X=(3756, 2595), y24=(3756, 24), features=2595
[INFO] smiles_col = Canonical_SMILES
[INFO] detected Morgan cols  = 2048
[INFO] detected StructKG cols = 47
[INFO] Training features remain FULL. Morgan/StructKG are NOT removed from model training.
[INFO] Beeswarm features also remain FULL. No feature is hidden in plotting.
[INFO] beeswarm candidate features = 2595
[INFO] features hidden only in beeswarm = 0

[TRAIN+SHAP] alcoholic (1/24)

[TRAIN+SHAP] aldehydic (2/24)

[TRAIN+SHAP] almond (3/24)

[TRAIN+SHAP] aromatic (4/24)

[TRAIN+SHAP] burnt (5/24)

[TRAIN+SHAP] cabbage (6/24)

[TRAIN+SHAP] cheesy (7/24)

[TRAIN+SHAP] cherry (8/24)

[TRAIN+SHAP] chocolate (9/24)

[TRAIN+SHAP] ethereal (10/24)

[TRAIN+SHAP] fishy (11/24)

[TRAIN+SHAP] fruity (12/24)

[TRAIN+SHAP] garlic (13/24)

[TRAIN+SHAP] grassy (14/24)

[TRAIN+SHAP] green (15/24)

[TRAIN+SHAP] ketonic (16/24)

[TRAIN+SHAP] musty (17/24)

[TRAIN+SHAP] pungent (18